# mAbs Animal-Study Pipeline (Colab)

End-to-end build of a structured table of mAb animal-study findings
from a PubMed query, plus per-stage eval runners. Everything — every
stage, every eval, every input (query, screening criteria, models,
extraction schema) — lives in this notebook and is editable below.

**Order:** Stage 1 -> Eval 1 -> Stage 2 -> Eval 2 -> ... -> Stage 6 -> Eval 6 -> Summary.

**What gets written to disk:** each stage writes under
`stage_NN/data/` (production artifacts) and `stage_NN/eval/` (graded
artifacts) in the notebook's working directory. Inspect those files
between stages if you want.


## 1. Install dependencies

`openai` for the LLM stages and evals; `pandas` for result tables;
the `liteparse` Node CLI for PDF -> text in Stage 4.


In [ ]:
%pip install -q openai>=1.40 pandas
!npm i -g @llamaindex/liteparse 2>&1 | tail -n 3


## 2. Set `OPENAI_API_KEY`

Stages 2 + 5 and the LLM evals (eval 1, 2, 5) need OpenAI. Stage 2
falls back to a regex emulator without a key; everything else either
runs offline or skips cleanly.

In Colab: store the key under Secrets (key icon in the left sidebar)
as `OPENAI_API_KEY`, then run this cell. Outside Colab: set the env
var before launching Jupyter.


In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("OPENAI_API_KEY loaded from Colab Secrets.")
    else:
        print("WARN: OPENAI_API_KEY not set in Colab Secrets — LLM stages will be skipped.")
except ImportError:
    if os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY found in environment.")
    else:
        print("WARN: OPENAI_API_KEY not set — LLM stages will be skipped.")


## 3. Configuration

All editable inputs live here. Change values, then **Run all** (or
re-run the cells below). The rest of the notebook reads these
variables directly.


In [ ]:
# ---------------------------------------------------------------------------
# Stage 1 — PubMed search
# ---------------------------------------------------------------------------

# PubMed esearch query. The workshop's canonical query targets
# mAb development + in-vivo arms + 2025-2026 publication date.
QUERY = (
    '("monoclonal antibody"[Title/Abstract] OR mAb[Title/Abstract] '
    'OR "monoclonal antibodies"[Title/Abstract]) '
    'AND (pharmacokinetic*[Title/Abstract] OR toxicology[Title/Abstract] '
    'OR toxicity[Title/Abstract] OR immunogenicity[Title/Abstract] '
    'OR biodistribution[Title/Abstract]) '
    'AND (cynomolgus[Title/Abstract] OR "non-human primate"[Title/Abstract] '
    'OR NHP[Title/Abstract] OR mouse[Title/Abstract] '
    'OR rat[Title/Abstract] OR rodent[Title/Abstract]) '
    'AND ("2025"[Date - Publication] : "2026"[Date - Publication])'
)

N = 30                                     # max PMIDs from esearch
TOOL  = "ar-bic-2026-workshop"             # NCBI identification
EMAIL = "workshop@example.org"

# ---------------------------------------------------------------------------
# Stage 2 — screen abstracts
# ---------------------------------------------------------------------------

SCREENER_MODEL = "gpt-5.4-nano"
SCREEN_SLEEP_SECONDS = 1.0

CRITERIA = """\
Include: primary research papers (2025-2026) reporting at least one
in-vivo mammalian study arm in support of monoclonal antibody (mAb)
development. Eligible study arms include:
  - pharmacokinetics (PK) or toxicokinetics
  - single-dose or repeat-dose toxicology
  - immunogenicity / anti-drug antibody (ADA) assessment
  - tissue biodistribution
  - tissue cross-reactivity confirmed in vivo

Eligible species: mouse, rat, cynomolgus monkey, rhesus monkey, dog,
rabbit, minipig.

Exclude:
  - reviews, meta-analyses, perspectives, commentaries, editorials
  - papers reporting only in-vitro binding, cell-line, or PBMC work
    with no in-vivo arm
  - veterinary mAb studies (animal as patient, not as preclinical model)
  - discovery-stage efficacy-only papers using mouse xenograft tumor
    models with no PK/tox/immunogenicity arm (mouse xenograft efficacy
    alone is not the reducible step we are studying)
  - mAb-conjugate papers where the conjugate (radioligand, toxin) is
    the primary subject and the antibody is incidental
  - case reports of mAb adverse events in patients
"""

# ---------------------------------------------------------------------------
# Stage 5 — structured extraction
# ---------------------------------------------------------------------------

EXTRACTOR_MODEL = "gpt-5.4-nano"
EXTRACT_SLEEP_SECONDS = 1.0

# Extraction contract.
SCHEMA = """\
{
  "pmid": "string",
  "source_type": "fulltext | abstract-only",
  "first_author": "string",
  "year": "integer",
  "mab_name": "string | null",
  "target": "string",
  "format": "IgG1 | IgG2 | IgG3 | IgG4 | bispecific | ADC | Fab | Fc-fusion | other",
  "development_stage": "discovery | lead optimization | IND-enabling | clinical translation | post-approval",
  "regulatory_context": "none-stated | IND-supporting | BLA-supporting | post-marketing",
  "threeRs_mentioned": "boolean",
  "author_reduction_recommendation": "string | null",
  "animal_arms": [
    {
      "species": "mouse | rat | cynomolgus | rhesus | dog | rabbit | minipig | other",
      "n_animals": "integer | null",
      "study_type": "PK | single-dose tox | repeat-dose tox | immunogenicity | biodistribution | efficacy | TCR",
      "duration_days": "integer | null",
      "species_justification": "pharmacological relevance | regulatory expectation | historical precedent | not stated",
      "cross_reactivity_evidence": "in-vitro binding shown | sequence homology only | not addressed",
      "endpoints_unique_to_animal": "string | null",
      "concurrent_nam": "string | null"
    }
  ],
  "nams_discussed": [
    {
      "method": "string",
      "context": "future work | limitation discussion | literature comparison"
    }
  ]
}
"""

# ---------------------------------------------------------------------------
# Eval — LLM grader / generator models
# ---------------------------------------------------------------------------

EVAL_MODEL     = "gpt-5.4-nano"            # grader (cheap, dominates cost)
EVAL_GEN_MODEL = "gpt-5.4-mini"            # question generator (smarter)
EVAL_05_MODEL  = "gpt-5.4-mini"            # eval_05_llm grader (mini-on-mini)

EVAL_02_N_QUESTIONS = 5                    # T/F per record in eval_02_llm
EVAL_05_N_QUESTIONS = 10                   # T/F per paper in eval_05_llm
EVAL_05_N_PAPERS    = 2                    # papers to grade in eval_05_llm

print("Config loaded.")
print(f"  Query: {QUERY[:80]}...")
print(f"  N PMIDs: {N}")
print(f"  Screener: {SCREENER_MODEL}  |  Extractor: {EXTRACTOR_MODEL}")
print(f"  Eval grader: {EVAL_MODEL}  |  Eval generator: {EVAL_GEN_MODEL}")


## 4. Helpers

Small utilities used across stages: section headers, score
bookkeeping, and a DataFrame display helper.


In [ ]:
import json
import os
from contextlib import contextmanager

try:
    import pandas as pd
    from IPython.display import display, Markdown
    _HAS_DISPLAY = True
except ImportError:
    _HAS_DISPLAY = False


def section(title, sub=""):
    """Print a visible section header in cell output."""
    bar = "=" * 72
    print(f"\n{bar}\n{title}")
    if sub:
        print(sub)
    print(bar)


@contextmanager
def step(label):
    """Print a 'running …' / 'done' pair so progress is visible."""
    print(f"\n[{label}] running ...")
    try:
        yield
    finally:
        print(f"[{label}] done.")


def write_score(stage, key, passed, total):
    """Merge {passed, total, percent} under `key` into stage/eval/score.json."""
    os.makedirs(f"{stage}/eval", exist_ok=True)
    path = f"{stage}/eval/score.json"
    data = {}
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
    pct = round(100.0 * passed / total, 1) if total else 0.0
    data[key] = {"passed": passed, "total": total, "percent": pct}
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"  score: {stage} {key} = {passed}/{total} ({pct}%)")


def show_df(df, caption=None, max_rows=12):
    """Render a DataFrame nicely if IPython is available; otherwise print."""
    if _HAS_DISPLAY:
        if caption:
            display(Markdown(f"**{caption}**"))
        display(df.head(max_rows))
    else:
        if caption:
            print(caption)
        print(df.head(max_rows).to_string(index=False))


def need_openai():
    """Return True if OPENAI_API_KEY is set; print a skip notice otherwise."""
    if os.environ.get("OPENAI_API_KEY"):
        return True
    print("  SKIP: no OPENAI_API_KEY in environment.")
    return False


## Stage 1 — PubMed search + metadata

Two NCBI E-utilities calls: `esearch` (query → PMIDs, capped at `N`)
then `efetch` (PMIDs → full metadata: title, abstract, authors,
journal, year, pub_types). Uses `.itertext()` so inline XML children
(`<i>`, `<sub>`, `<sup>`) survive — a common silent-truncation bug
when reading PubMed XML naively.

Writes `stage_01/data/pmids.json` and displays the first records.


In [ ]:
import json
import os
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

section("Stage 1 — PubMed search", f"query → {N} PMIDs, then efetch metadata")

STAGE = "stage_01"
DATA = f"{STAGE}/data"
os.makedirs(DATA, exist_ok=True)
HEADERS = {"User-Agent": "ar-bic-2026/0.1"}

# ---- esearch ----
with step("esearch"):
    q = urllib.parse.urlencode({
        "db": "pubmed", "term": QUERY, "retmax": N,
        "retmode": "json", "sort": "date",
        "tool": TOOL, "email": EMAIL,
    })
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?{q}"
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=30) as r:
        pmids = json.load(r)["esearchresult"]["idlist"]
    assert pmids, "esearch returned zero PMIDs — check the QUERY in cell 3"
    assert all(p.isdigit() for p in pmids), "non-numeric PMID returned"
    print(f"  {len(pmids)} PMIDs returned")


# ---- efetch metadata ----
def _text(el):
    if el is None:
        return ""
    return "".join(el.itertext()).strip()


def parse_record(art):
    pmid = art.findtext(".//PMID") or ""
    title = _text(art.find(".//ArticleTitle"))

    ab_parts = []
    for ab in art.findall(".//Abstract/AbstractText"):
        text = _text(ab)
        label = ab.get("Label")
        if not text:
            continue
        ab_parts.append(f"{label}: {text}" if label else text)
    abstract = " ".join(ab_parts).strip()

    authors = []
    for a in art.findall(".//AuthorList/Author"):
        coll = a.findtext("CollectiveName") or ""
        ln = a.findtext("LastName") or ""
        fn = a.findtext("ForeName") or a.findtext("Initials") or ""
        name = coll.strip() if coll else f"{ln} {fn}".strip()
        if name:
            authors.append(name)

    journal = (
        art.findtext(".//Journal/Title")
        or art.findtext(".//Journal/ISOAbbreviation")
        or ""
    ).strip()

    year_str = (
        art.findtext(".//Journal/JournalIssue/PubDate/Year")
        or art.findtext(".//Journal/JournalIssue/PubDate/MedlineDate")
        or ""
    )
    try:
        year = int(year_str[:4])
    except (ValueError, TypeError):
        year = None

    pub_types = [t.text for t in art.findall(".//PublicationType") if t.text]

    return {
        "pmid": pmid, "title": title, "abstract": abstract,
        "authors": authors, "first_author": authors[0] if authors else "",
        "journal": journal, "year": year, "pub_types": pub_types,
    }


with step("efetch metadata"):
    q = urllib.parse.urlencode({
        "db": "pubmed", "id": ",".join(pmids), "retmode": "xml",
        "tool": TOOL, "email": EMAIL,
    })
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?{q}"
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=60) as r:
        xml = r.read().decode("utf-8")
    records = [parse_record(a) for a in ET.fromstring(xml).findall(".//PubmedArticle")]
    by_pmid = {r["pmid"]: r for r in records}
    records = [by_pmid[p] for p in pmids if p in by_pmid]
    print(f"  {len(records)} metadata records parsed")


out = f"{DATA}/pmids.json"
with open(out, "w") as f:
    json.dump({"query": QUERY, "n_requested": N, "pmids": pmids, "records": records}, f, indent=2)

assert len(records) == len(pmids), f"efetch returned {len(records)} for {len(pmids)} PMIDs"
for r in records:
    assert r["pmid"], "record missing PMID"
    assert r["title"], f"{r['pmid']}: empty title"

print(f"\nOK — {len(pmids)} PMIDs + {len(records)} metadata records → {out}")

# ---- display ----
import pandas as pd
df = pd.DataFrame([{
    "pmid": r["pmid"],
    "year": r["year"],
    "first_author": r["first_author"],
    "journal": r["journal"][:40],
    "title": r["title"][:80],
} for r in records])
show_df(df, caption=f"Stage 1 results — top {min(12, len(df))} of {len(df)} records")


### Eval — Stage 1

Two halves:

1. **Script-only** (offline) — PMID-list shape + per-record metadata
   completeness, including an XML-tag-leak regex that catches the
   `.text`-vs-`.itertext()` truncation bug.
2. **LLM T/F** (needs `OPENAI_API_KEY`) — two formality questions per
   record on the top-10: "does this title / abstract look like
   properly formed PubMed prose?" Topic relevance is deliberately
   ignored — Stage 2 handles that.


In [ ]:
import datetime
import json
import os
import re

section("Eval Stage 1 — script checks", "deterministic re-checks of pmids.json")

XML_TAG_LEAK = re.compile(r"</?[a-zA-Z]")
STAGE = "stage_01"
os.makedirs(f"{STAGE}/eval", exist_ok=True)

REQUIRED_FIELDS = {"pmid", "title", "abstract", "authors", "first_author",
                   "journal", "year", "pub_types"}
THIS_YEAR = datetime.date.today().year

with open(f"{STAGE}/data/pmids.json") as f:
    doc = json.load(f)
pmids = doc["pmids"]
records = doc.get("records") or []

script_checks = {
    "pmids_non_empty": bool(pmids),
    "all_numeric": all(p.isdigit() for p in pmids),
    "no_duplicates": len(pmids) == len(set(pmids)),
    "respects_cap": len(pmids) <= doc.get("n_requested", len(pmids)),
    "records_match_pmids": [r["pmid"] for r in records] == pmids,
}

per_record_issues = []
n_fields_present = n_title_ok = n_abstract_long = n_abstract_no_xml_leak = 0
n_authors_non_empty = n_journal_ok = n_year_ok = n_pubtypes_ok = 0

for r in records:
    issues = []
    if set(r.keys()) >= REQUIRED_FIELDS: n_fields_present += 1
    else: issues.append(f"missing keys: {REQUIRED_FIELDS - set(r.keys())}")
    if r.get("title") and len(r["title"]) > 10: n_title_ok += 1
    else: issues.append("title missing or < 10 chars")
    abstract = r.get("abstract") or ""
    if len(abstract) > 200: n_abstract_long += 1
    if not XML_TAG_LEAK.search(abstract): n_abstract_no_xml_leak += 1
    else: issues.append("abstract contains XML-tag-shaped leakage")
    if r.get("authors"): n_authors_non_empty += 1
    if r.get("journal"): n_journal_ok += 1
    year = r.get("year")
    if isinstance(year, int) and 1990 <= year <= THIS_YEAR + 1: n_year_ok += 1
    else: issues.append(f"year out of range: {year!r}")
    if r.get("pub_types"): n_pubtypes_ok += 1
    else: issues.append("pub_types empty")
    if issues:
        per_record_issues.append({"pmid": r.get("pmid"), "issues": issues})

n = len(records)
script_checks.update({
    "all_records_have_required_fields": n_fields_present == n,
    "all_titles_substantive": n_title_ok == n,
    "no_xml_tag_leakage_in_abstracts": n_abstract_no_xml_leak == n,
    "most_abstracts_full_length": (n_abstract_long / n) >= 0.6 if n else True,
    "most_records_have_authors": (n_authors_non_empty / n) >= 0.9 if n else True,
    "all_journals_named": n_journal_ok == n,
    "all_years_in_range": n_year_ok == n,
    "all_records_have_pub_types": n_pubtypes_ok == n,
})

for k, v in script_checks.items():
    print(f"  {'OK  ' if v else 'FAIL'}  {k}")
if per_record_issues:
    print(f"\nPer-record issues ({len(per_record_issues)}):")
    for it in per_record_issues[:5]:
        print(f"  {it['pmid']}: {'; '.join(it['issues'])}")

with open(f"{STAGE}/eval/eval_script.json", "w") as f:
    json.dump({"script": script_checks, "per_record_issues": per_record_issues}, f, indent=2)

n_total = len(script_checks)
n_passed = sum(1 for v in script_checks.values() if v)
write_score(STAGE, "script", n_passed, n_total)


In [ ]:
# Eval 1 — LLM formality T/F on top-10 records (needs OPENAI_API_KEY)
section("Eval Stage 1 — LLM T/F", f"grader={EVAL_MODEL}, 2 questions × top-10")

if need_openai():
    from openai import OpenAI
    client = OpenAI()
    STAGE = "stage_01"
    os.makedirs(f"{STAGE}/eval", exist_ok=True)

    def grade(question, context):
        prompt = (
            "Answer with strictly TRUE or FALSE based only on the context. "
            "No scoring, no 'partial', no hedging, no explanation. "
            'Return JSON: {"answer": "TRUE" | "FALSE"}.\n\n'
            f"QUESTION:\n{question}\n\nCONTEXT:\n{context}"
        )
        resp = client.chat.completions.create(
            model=EVAL_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content).get("answer", "").upper() == "TRUE"

    with open(f"{STAGE}/data/pmids.json") as f:
        records = json.load(f).get("records") or []

    QUESTIONS = [
        ("title",
         "Does this string look like a properly formed PubMed article "
         "title — complete (not cut off mid-word or mid-phrase), readable, "
         "and free of garbled characters or placeholder content? Answer "
         "TRUE if formatting is sound, regardless of topic."),
        ("abstract",
         "Does this string look like properly formed PubMed abstract "
         "text — complete (not cut off mid-sentence or marked with '...' "
         "or '[truncated]'), readable as scientific prose, and free of "
         "obvious encoding or parsing artifacts? An empty abstract is "
         "FALSE; a short but coherent abstract is TRUE."),
    ]

    top = records[:10]
    items = []
    for i, rec in enumerate(top, 1):
        print(f"  [{i}/{len(top)}] formality T/F  pmid={rec.get('pmid')}")
        checks = []
        for field, question in QUESTIONS:
            value = rec.get(field, "") or ""
            ctx = f"FIELD: {field}\nVALUE:\n{value}"
            checks.append({"field": field, "q": question,
                           "answer": grade(question, ctx)})
        items.append({"pmid": rec.get("pmid"), "checks": checks})

    with open(f"{STAGE}/eval/eval_llm.json", "w") as f:
        json.dump({"ai": items}, f, indent=2)

    trues = sum(1 for it in items for c in it["checks"] if c["answer"])
    total = sum(len(it["checks"]) for it in items)
    print(f"\nAI T/F: {trues}/{total} TRUE across top-{len(top)} × {len(QUESTIONS)} questions")
    write_score(STAGE, "llm", trues, total)


## Stage 2 — Screen abstracts

For each record from Stage 1, decide `include` / `exclude` against
the criteria in cell 3. OpenAI when the key is set; the regex
emulator otherwise so the pipeline runs end-to-end without a key.

Writes `stage_02/data/screened.json`.


### Your task — write Stage 2

Implement Stage 2 in the empty code cell at the bottom of this
section.

To use Gemini: open Colab's **Gemini** panel (sparkle icon, top-right
of the toolbar). Copy the next cell's content (cell menu -> *Copy cell
content*, or double-click the cell, select-all, copy) and paste it
into Gemini as your prompt. Review the code Gemini gives you, paste
it into the empty cell below, then run.

Once it runs cleanly, run the eval cells below to validate your work.


Goal: screen each Stage 1 record for inclusion/exclusion.

Already defined in earlier cells: CRITERIA (str, the inclusion/
exclusion text), SCREENER_MODEL (str), SCREEN_SLEEP_SECONDS (float).
Helpers `section`, `show_df`. `stage_01/data/pmids.json` exists with
shape `{"records": [{"pmid", "title", "abstract", "pub_types", ...}]}`.

Use Python stdlib plus pandas; import `openai.OpenAI` only inside the
OpenAI branch.

Steps:
1. Load `stage_01/data/pmids.json` and extract `records`.

2. Define `screen_openai(rec)`:
   - Call `openai.OpenAI().chat.completions.create(...)` with
     `model=SCREENER_MODEL, temperature=0,
     response_format={"type": "json_object"}`.
   - Prompt content: ask the model to screen the abstract against
     CRITERIA and return strict JSON
     `{"verdict": "include"|"exclude",
       "rationale": "<one sentence naming the criterion>"}`.
     No markdown fences.
   - Embed CRITERIA, then `PMID: ... / Title: ... / Abstract: ...`.
   - Return `json.loads(resp.choices[0].message.content)`.

3. Define `screen_regex(rec)` - deterministic fallback:
   - `text = (title + " " + abstract).lower()`,
     `pt = " ".join(pub_types).lower()`.
   - Exclude if `pt` contains any of review/meta-analysis/editorial/
     comment.
   - Exclude if `re.search(r"\b(canine|feline|equine|bovine|porcine)\s+patient", text)` (veterinary mAb).
   - Detect signals (all regex, word-boundary):
     - `mab` = `r"\b(monoclonal antibody|monoclonal antibodies|mab|igg1|igg2|igg3|igg4|bispecific)\b"`
     - `species` = `r"\b(cynomolgus|rhesus|non[- ]human primate|nhp|mouse|mice|rat|rats|dog|canine|rabbit|minipig)\b"`
     - `study` = `r"\b(pharmacokinetic|toxicokinetic|toxicology|toxicity|immunogenicity|anti[- ]drug antibod(?:y|ies)|biodistribution|tissue cross[- ]reactivity)\b"`
     - `invivo` = `r"\b(in vivo|administered|dosed|injection)\b"`
   - `xenograft_only` = `xenograft` matches AND none of
     pharmacokinetic/toxicology/immunogenicity/biodistribution match.
   - Verdicts (in order): xenograft_only -> exclude; all four signals
     match -> include with rationale `"mAb + in-vivo species + IND-relevant study type present"`;
     else exclude with `"no in-vivo IND-relevant study signal found"`.

4. Pick the screener by `os.environ.get("OPENAI_API_KEY")`. Print
   which one is running (and the model name if OpenAI).

5. For each record, call the screener inside `try/except` (on
   exception return `{"verdict": "exclude",
   "rationale": f"screening error: {type(e).__name__}"}`).
   Build per-record output as
   `{pmid, title, abstract, pub_types, verdict, rationale}`.
   When the screener is `screen_openai`, sleep
   `SCREEN_SLEEP_SECONDS` between calls.

6. Write `stage_02/data/screened.json` (list of dicts, indent=2).
   Assertions: every record has pmid, verdict in {include, exclude},
   non-empty rationale.

7. Print `"OK - X/Y included -> stage_02/data/screened.json"`.
   Display a DataFrame `{pmid, verdict, rationale[:70], title[:60]}`
   via `show_df`.

Per-iteration progress: `print(f"  [{i}/{len(records)}] screen  pmid={rec['pmid']}")` at the top. No tqdm.


In [ ]:
# Your Stage 2 implementation goes here.
# Paste Gemini's output (or your own code) into this cell.


### Eval — Stage 2

Script half checks the JSON shape (PMID + verdict ∈ {include,exclude}
+ non-empty rationale). LLM half is generate-then-verify with seed=42:
sample 5 include + 5 exclude, generator drafts T/F questions tagged
`include` or `exclude`, grader answers from criteria + abstract, then
we infer the verdict by polarity rules and compare to the screener.


In [ ]:
section("Eval Stage 2 — script checks", "verdict + rationale shape")

import json, os

STAGE = "stage_02"
os.makedirs(f"{STAGE}/eval", exist_ok=True)

with open(f"{STAGE}/data/screened.json") as f:
    recs = json.load(f)

script_checks = {
    "every_record_has_pmid": all(r.get("pmid") for r in recs),
    "verdicts_valid": all(r.get("verdict") in {"include", "exclude"} for r in recs),
    "rationales_non_empty": all((r.get("rationale") or "").strip() for r in recs),
}

for k, v in script_checks.items():
    print(f"  {'OK  ' if v else 'FAIL'}  {k}")

with open(f"{STAGE}/eval/eval_script.json", "w") as f:
    json.dump({"script": script_checks}, f, indent=2)

write_score(STAGE, "script",
            sum(1 for v in script_checks.values() if v),
            len(script_checks))


In [ ]:
# Eval 2 — generate-then-verify criteria-decomposition rubric
section("Eval Stage 2 — LLM rubric",
        f"generator={EVAL_GEN_MODEL}, grader={EVAL_MODEL}, 5 incl + 5 excl @ seed=42")

if need_openai():
    import json, os, random
    from openai import OpenAI
    client = OpenAI()

    STAGE = "stage_02"
    N_QUESTIONS = EVAL_02_N_QUESTIONS

    with open(f"{STAGE}/data/screened.json") as f:
        recs = json.load(f)

    def llm_json(prompt, model):
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content)

    def generate_questions(rec, n):
        prompt = (
            "You are designing a quality-check for a literature-screening "
            "decision. The criteria below define which papers belong in a "
            "systematic review of mAb animal studies.\n\n"
            f"Read the CRITERIA, then read the ABSTRACT. Generate {n} "
            "self-contained TRUE/FALSE questions, each targeting ONE "
            "specific criterion. Mix include- and exclude-side criteria.\n\n"
            'Each question carries a category: "include" (TRUE = inclusion '
            'criterion met) or "exclude" (TRUE = exclusion criterion fires).\n\n'
            "Frame each question as a single positive assertion. No "
            "\"rather than\" / \"as opposed to\" — they confuse polarity.\n\n"
            'Return strict JSON: {"items": [{"q": "...", "category": '
            '"include"|"exclude"}, ...]}. No markdown fences.\n\n'
            f"CRITERIA:\n{CRITERIA}\n\n"
            f"ABSTRACT:\nPMID: {rec['pmid']}\nTitle: {rec['title']}\n"
            f"Abstract: {rec['abstract']}"
        )
        return llm_json(prompt, EVAL_GEN_MODEL).get("items", [])

    def grade(question, rec):
        ctx = (
            f"CRITERIA:\n{CRITERIA}\n\n"
            f"ABSTRACT:\nPMID: {rec['pmid']}\nTitle: {rec['title']}\n"
            f"Abstract: {rec['abstract']}"
        )
        prompt = (
            "Answer with strictly TRUE or FALSE based only on the context. "
            'Return JSON: {"answer": "TRUE" | "FALSE"}.\n\n'
            f"QUESTION:\n{question}\n\nCONTEXT:\n{ctx}"
        )
        return llm_json(prompt, EVAL_MODEL).get("answer", "").upper() == "TRUE"

    def infer_verdict(checks):
        any_exclude_triggered = any(c["actual"] for c in checks if c["category"] == "exclude")
        any_include_missing = any(not c["actual"] for c in checks if c["category"] == "include")
        return "exclude" if (any_exclude_triggered or any_include_missing) else "include"

    rng = random.Random(42)
    includes = [r for r in recs if r.get("verdict") == "include"]
    excludes = [r for r in recs if r.get("verdict") == "exclude"]
    inc_sample = rng.sample(includes, min(5, len(includes)))
    exc_sample = rng.sample(excludes, min(5, len(excludes)))
    sample = inc_sample + exc_sample
    print(f"Sampled {len(inc_sample)} include + {len(exc_sample)} exclude")

    items = []
    for i, rec in enumerate(sample, 1):
        print(f"  [{i}/{len(sample)}] generate+grade  pmid={rec['pmid']}  (screener={rec['verdict']})")
        qcat = generate_questions(rec, N_QUESTIONS)
        checks = []
        for ix in qcat:
            q = ix.get("q", "")
            cat = ix.get("category", "include")
            actual = grade(q, rec)
            checks.append({"q": q, "category": cat, "actual": actual})
        inferred = infer_verdict(checks)
        is_pass = inferred == rec["verdict"]
        print(f"  {rec['pmid']} ({rec['verdict']} → {inferred})  "
              f"{'PASS' if is_pass else 'FAIL'}")
        items.append({"pmid": rec["pmid"], "verdict": rec["verdict"],
                      "inferred": inferred, "pass": is_pass, "checks": checks})

    with open(f"{STAGE}/eval/eval_llm.json", "w") as f:
        json.dump({"ai_per_record": items}, f, indent=2)

    passed = sum(1 for it in items if it["pass"])
    print(f"\n{passed}/{len(items)} PASS")
    write_score(STAGE, "llm", passed, len(items))
